In [3]:
import pandas as pd
import numpy as np

In [ ]:
"""
df = pd.read_csv(
    'data/train.csv', 
    parse_dates=['timestamp'], 
    date_format='%H:%M:%S'
)
"""

In [ ]:
#print(df.head(2).T)

# TODO  
* matrice correlazione feature (più correlate = da eliminare perché meno informative)  
* n° task per tipologia per snapshot  
* n° task per singolo aereo per snapshot  
* n° aeromobili distinti per snapshot  
* n° aeromobili sotto controllo  
* età massima task per snapshot  
* quante scelte diverse per ogni snapshot (ogni 3 secondi, sample_id)

In [ ]:
def preprocess_memory_efficient(df, is_train=True):
    # 1. Lista colonne di metadati e ID ad alta cardinalità da ESCLUDERE dalle feature
    #metadata_cols = ['sample_id', 'row_index', 'session_id', 'atco_id', 'aircraft_id', 'aircraft']
    metadata_cols = [
    'sample_id',
    'row_index',
    'session',
    'atco',
    'aircraft_id',
    'aircraft',
    'timestamp',
    'task_name'
    ]
    target_col = 'target_index'

    df['task_index'] = df['task_index'].astype('category')
    
    # 2. Downcasting immediato dei numeri decimali a float32 (risparmia il 50% di RAM)
    float_cols = df.select_dtypes(include=['float64']).columns
    df[float_cols] = df[float_cols].astype(np.float32)
    
    # 3. Downcasting degli interi
    int_cols = df.select_dtypes(include=['int64']).columns
    for c in int_cols:
        if c not in ['sample_id', 'row_index']:
            df[c] = df[c].astype(np.int32)
            
    # 4. Tipi booleani
    bool_cols = df.select_dtypes(include=['bool']).columns
    df[bool_cols] = df[bool_cols].astype(np.int8)

    # 5. Gestione categoriche sicura (solo colonne con poche categorie distinte, es. task_name)
    # Ignora colonne con troppi valori unici che causano MemoryError
    for col in df.select_dtypes(include=['object', 'string']).columns:
        if col not in metadata_cols:
            if df[col].nunique() < 100:  # ad esempio task_name ha 14 classi
                df[col] = df[col].astype('category')
            else:
                # Se ha troppi valori unici ed è una stringa, la escludiamo
                metadata_cols.append(col)

    # 6. Feature di Gruppo: info. di contesto sullo snapshot
    # rank_time_to_sector: ranking del task rispetto all'imminenza
    # diff_from_min_time: distanza del task corrente rispetto al task più imminente

    '''
    if 'time_to_sector' in df.columns:
        abs_time = df['time_to_sector'].abs().astype(np.float32)
        df['rank_time_to_sector'] = df.groupby('sample_id')['time_to_sector'].rank(method='min').astype(np.int16)
        min_time = df.groupby('sample_id')['time_to_sector'].transform('min').astype(np.float32)
        df['diff_from_min_time'] = (abs_time - min_time).astype(np.float32)
        del abs_time, min_time
    '''
    # 7. Isolamento Feature
    exclude = metadata_cols + ([target_col] if target_col in df.columns else [])
    feature_cols = [c for c in df.columns if c not in exclude]
    
    return df, feature_cols

# ESECUZIONE
#train_df, feature_cols = preprocess_memory_efficient(pd.read_csv('data/train.csv'), is_train=True)



In [10]:
print(train_df.head(4).T)
#print(feature_cols)

                              0           1           2           3
sample_id                     0           0           0           0
seq_len                      29          29          29          29
row_index                     0           1           2           3
target_index                  0           0           0           1
task_index                    4           2          12           2
...                         ...         ...         ...         ...
l_frac_type            0.413793    0.448276    0.137931    0.448276
l_twin_AE                     1           1           1           1
l_rank_age_type               1           1           1           1
rank_time_to_sector          17          17           9           9
diff_from_min_time   576.351013  576.351013  308.455994  308.455994

[98 rows x 4 columns]
